<a href="https://colab.research.google.com/github/heyanugrah/Transformers/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Transformer- Encoder

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# 1. Define the Vocabulary (Adding '[sep]')
new_vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3, 'this': 4, 'is': 5, 'a': 6, 'positive': 7, 'sentence': 8, 'learning': 9, 'about': 10, 'transformers': 11, 'fun': 12, 'i': 13, 'hate': 14, 'am': 15, 'so': 16, 'disappointed': 17, 'really': 18, 'enjoyed': 19, 'movie': 20, 'wonderful': 21, 'day': 22, 'for': 23, 'walk': 24, 'product': 25, 'excellent!': 26, 'absolutely': 27, 'dreadful': 28, 'service': 29, 'what': 30, 'terrible': 31, 'experience': 32, 'feeling': 33, 'very': 34, 'happy': 35, 'today': 36, 'boy': 37, 'anugrah': 38}

# 2. Define the Reverse Vocabulary
idx_to_word = {i: word for word, i in new_vocab.items()}

vocab_size = len(new_vocab)

print(f"Vocabulary size is: {vocab_size}")

Vocabulary size is: 39


### Modular Tokenization Functions

To improve clarity and reusability, let's break down the tokenization process into several distinct functions:

1.  `_add_special_tokens_and_segments`: Adds `<cls>` and `[sep]` tokens and initializes segment IDs.
2.  `_convert_to_ids`: Converts token strings to their numerical IDs.
3.  `_pad_ids_and_segments`: Handles padding for both token IDs and segment IDs.
4.  `_create_attention_mask_from_ids`: Generates the attention mask from padded token IDs.
5.  `prepare_transformer_inputs`: An orchestrating function that calls the above helpers to produce all necessary inputs for a Transformer model.

In [3]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [4]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

### Demonstration of the new tokenization pipeline

In [5]:


input_text = "anugrah is learning"
max_seq_len = 10 # maximum token need to be created

# Prepare inputs using the new modular function
input_ids, attention_mask_new, token_type_ids_new = prepare_transformer_inputs(
    input_text, new_vocab, max_seq_len
)

print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask_new}")
print(f"Token Type IDs: {token_type_ids_new}")

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Mask shape: {attention_mask_new.shape}")
print(f"Token Type IDs shape: {token_type_ids_new.shape}")

Input Text: 'anugrah is learning'
Input IDs: tensor([[ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0]])
Attention Mask: tensor([[[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs shape: torch.Size([1, 10])
Attention Mask shape: torch.Size([1, 1, 1, 10])
Token Type IDs shape: torch.Size([1, 10])


### Create and apply embedding

In [6]:
d_model = 8 #vector dimension for each word

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

embedded_input = embedding(input_ids)
print(f"Shape of embedded input: {embedded_input.shape}")

Shape of embedded input: torch.Size([1, 10, 8])


In [7]:
print(embedded_input)

tensor([[[-0.3420,  0.1170, -0.0727,  1.6669, -1.2632, -0.4184,  0.8575,
          -0.6476],
         [ 1.1646,  1.0374, -0.0597, -0.0381, -0.7792,  0.6496, -0.0499,
           0.2087],
         [ 0.1067,  0.0804, -0.5878, -0.8523,  2.1130,  0.9933, -0.4190,
           0.5442],
         [ 0.6254,  0.3953,  0.3151, -0.3130,  0.3219, -1.8996, -0.6867,
           0.8848],
         [-0.3420,  0.1170, -0.0727,  1.6669, -1.2632, -0.4184,  0.8575,
          -0.6476],
         [-0.4553,  0.5956,  0.5885, -0.3486, -1.9994,  0.0458, -0.2582,
           0.9702],
         [-0.4553,  0.5956,  0.5885, -0.3486, -1.9994,  0.0458, -0.2582,
           0.9702],
         [-0.4553,  0.5956,  0.5885, -0.3486, -1.9994,  0.0458, -0.2582,
           0.9702],
         [-0.4553,  0.5956,  0.5885, -0.3486, -1.9994,  0.0458, -0.2582,
           0.9702],
         [-0.4553,  0.5956,  0.5885, -0.3486, -1.9994,  0.0458, -0.2582,
           0.9702]]], grad_fn=<EmbeddingBackward0>)


In [8]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nEncodings for each token in 'anugrah is learning':")
# Iterate through the input_ids and corresponding embedded vectors
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    embedding_vector = embedded_input.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Embedding: {embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Encodings for each token in 'anugrah is learning':
Token: '<unk>' (ID: 1) -> Embedding: [-0.34203505516052246, 0.1170378252863884, -0.07270318269729614, 1.666906476020813, -1.2632460594177246, -0.41837558150291443, 0.8575294017791748, -0.6476019620895386]
Token: 'anugrah' (ID: 38) -> Embedding: [1.1646440029144287, 1.0373879671096802, -0.05968241021037102, -0.03809936344623566, -0.7792027592658997, 0.6496221423149109, -0.04993666335940361, 0.20873261988162994]
Token: 'is' (ID: 5) -> Embedding: [0.1067444235086441, 0.08040285110473633, -0.587821364402771, -0.8522621989250183, 2.113001823425293, 0.993332028388977, -0.419040322303772, 0.5441792607307434]
Token: 'learning' (ID: 9) -> Embedding: [0.6253619194030762, 0.3953385353088379, 0.31514376401901245, -0.3129793405532837, 0.321918785572052, -1.8995729684829712, -0.6866507530212402, 0.8847966194152832]
Token: '<unk>' (ID: 1) -> Embedding: [-0.

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Instantiate Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_seq_len)

# Apply positional encoding to the embedded input
embedded_with_pos = pos_encoder(embedded_input)

print(f"Shape of embedded input with positional encoding: {embedded_with_pos.shape}")

Shape of embedded input with positional encoding: torch.Size([1, 10, 8])


In [10]:
print(embedded_with_pos[0][1],embedded_with_pos.shape) #PE for anugrah (1,batch,10 total , 8 dimension)

tensor([ 2.2290,  1.7530,  0.0446,  1.0632, -0.8547,  1.8329, -0.0544,  1.3430],
       grad_fn=<SelectBackward0>) torch.Size([1, 10, 8])


In [11]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nCombined Embeddings (Word Embedding + Positional Encoding) for each token:")
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    combined_embedding_vector = embedded_with_pos.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Combined Embedding: {combined_embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Combined Embeddings (Word Embedding + Positional Encoding) for each token:
Token: '<unk>' (ID: 1) -> Combined Embedding: [-0.3800389766693115, 1.2411531209945679, -0.0, 2.9632294178009033, -0.0, 0.6462493538856506, 0.9528104662895203, 0.3915534019470215]
Token: 'anugrah' (ID: 38) -> Combined Embedding: [2.2290167808532715, 1.752989411354065, 0.044612228870391846, 1.063227653503418, -0.854669988155365, 1.8328579664230347, -0.054374076426029205, 1.3430358171463013]
Token: 'is' (ID: 5) -> Combined Embedding: [1.12893545627594, -0.37304890155792236, -0.4323911964893341, 0.14200489223003387, 2.3700006008148193, 0.0, -0.4633781611919403, 1.7157526016235352]
Token: 'learning' (ID: 9) -> Combined Embedding: [0.8516466617584229, -0.6607266664505005, 0.6785155534744263, 0.7137302160263062, 0.391015887260437, -1.0000255107879639, -0.759611964225769, 2.0942134857177734]
Token: '<unk>' (ID: 1) -> Combined

In [12]:
token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types: 0 for sentence A, 1 for sentence B

'''
Segment Embedding

Tells the model which sentence a token belongs to.

Example:

Sentence A: I love AI
Sentence B: It is powerful

BERT input:

[CLS] I love AI [SEP] It is powerful [SEP]
'''

# Get segment embeddings for our token_type_ids
segment_embeddings = token_type_embedding(token_type_ids_new)

# Add the segment embeddings to the combined word and positional embeddings
final_transformer_input = embedded_with_pos + segment_embeddings

print(f"Shape of token_type_ids: {token_type_ids_new.shape}")
print(f"Shape of segment embeddings: {segment_embeddings.shape}")
print(f"Shape of final input to Transformer: {final_transformer_input.shape}")

print("\nSegment Embeddings:")
print(segment_embeddings)

Shape of token_type_ids: torch.Size([1, 10])
Shape of segment embeddings: torch.Size([1, 10, 8])
Shape of final input to Transformer: torch.Size([1, 10, 8])

Segment Embeddings:
tensor([[[ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
           0.1771],
         [ 0.9754,  2.7996, -1.6230, -1.6211,  1.3660, -0.6481, -0.1711,
     

In [13]:
print('final transformer input')
print(final_transformer_input)

final transformer input
tensor([[[ 5.9540e-01,  4.0408e+00, -1.6230e+00,  1.3421e+00,  1.3660e+00,
          -1.8876e-03,  7.8168e-01,  5.6867e-01],
         [ 3.2045e+00,  4.5526e+00, -1.5784e+00, -5.5792e-01,  5.1134e-01,
           1.1847e+00, -2.2551e-01,  1.5201e+00],
         [ 2.1044e+00,  2.4266e+00, -2.0554e+00, -1.4791e+00,  3.7360e+00,
          -6.4814e-01, -6.3451e-01,  1.8929e+00],
         [ 1.8271e+00,  2.1389e+00, -9.4452e-01, -9.0741e-01,  1.7570e+00,
          -1.6482e+00, -9.3075e-01,  2.2713e+00],
         [-2.4549e-01,  2.2034e+00, -1.2711e+00,  1.2544e+00,  6.8378e-03,
          -2.7764e-03, -1.7114e-01,  5.6866e-01],
         [-5.9595e-01,  3.7765e+00, -1.6230e+00, -1.0334e+00, -8.0004e-01,
           5.1251e-01, -4.5246e-01,  2.3663e+00],
         [ 1.5906e-01,  2.7996e+00, -3.4180e-01, -1.0914e+00, -7.8894e-01,
          -6.4814e-01, -4.5135e-01,  2.3663e+00],
         [ 1.1995e+00,  4.2990e+00, -2.5338e-01, -1.1586e+00,  1.3660e+00,
           5.1117e-01, -4.

In [14]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply padding mask to the scores. Positions where mask == 0 are set to -inf.
            # This prevents attention to padded tokens in the input sequence.
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output

        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# # --- Execution Code ---
# encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
# encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

# print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
# print(f"Encoder Layer Output Shape: {encoder_output.shape}")
# print("\nFirst few values of the output:")
# print(encoder_output[0, 0, :])

In [15]:
'''
Input Target Sentence
        ↓
Embedding
        ↓
Positional Encoding
        ↓
Masked Self Attention
(Q,K,V from Decoder Input)
        ↓
Add + Norm
        ↓
Cross Attention
(Q from Decoder)

(K,V from Encoder Output)
        ↓
Add + Norm
        ↓
Feed Forward Network
        ↓
Add + Norm
        ↓
Decoder Output
'''

class TransformerDecoderLayer(nn.Module):

    def __init__(self,
                 d_model,
                 num_heads,
                 ffn_hidden_dim,
                 dropout_rate=0.1):

        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # ===============================
        # 1. Masked Self Attention
        # ===============================

        self.self_Wq = nn.Linear(d_model, d_model)
        self.self_Wk = nn.Linear(d_model, d_model)
        self.self_Wv = nn.Linear(d_model, d_model)

        self.self_Wo = nn.Linear(d_model, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout_rate)

        # ===============================
        # 2. Encoder Decoder Attention
        # ===============================

        self.cross_Wq = nn.Linear(d_model, d_model)

        self.cross_Wk = nn.Linear(d_model, d_model)
        self.cross_Wv = nn.Linear(d_model, d_model)

        self.cross_Wo = nn.Linear(d_model, d_model)

        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout_rate)

        # ===============================
        # 3. Feed Forward Network
        # ===============================

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout_rate)

    def split_heads(self, x):

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        return x.transpose(1, 2)

    def attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)

        return output

    def forward(self,
                decoder_input,
                encoder_output,
                src_mask,
                tgt_mask):

        batch_size = decoder_input.shape[0]
        decoder_seq_len = decoder_input.shape[1]

        # ==========================================
        # STEP 1 : Masked Self Attention
        # ==========================================

        Q = self.self_Wq(decoder_input)
        K = self.self_Wk(decoder_input)
        V = self.self_Wv(decoder_input)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        attention_output = self.attention(
            Q,
            K,
            V,
            tgt_mask
        )

        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                decoder_seq_len,
                self.d_model
            )
        )

        attention_output = self.self_Wo(
            attention_output
        )

        attention_output = self.dropout1(
            attention_output
        )

        decoder_output = self.norm1(
            decoder_input + attention_output
        )

        # ==========================================
        # STEP 2 : Encoder Decoder Attention
        # ==========================================

        encoder_seq_len = encoder_output.shape[1]

        Q = self.cross_Wq(
            decoder_output
        )

        K = self.cross_Wk(
            encoder_output
        )

        V = self.cross_Wv(
            encoder_output
        )

        Q = self.split_heads(Q)
        K = K.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        cross_output = self.attention(
            Q,
            K,
            V,
            src_mask
        )

        cross_output = (
            cross_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                decoder_seq_len,
                self.d_model
            )
        )

        cross_output = self.cross_Wo(
            cross_output
        )

        cross_output = self.dropout2(
            cross_output
        )

        decoder_output = self.norm2(
            decoder_output + cross_output
        )

        # ==========================================
        # STEP 3 : Feed Forward Network
        # ==========================================

        ffn_output = self.ffn(
            decoder_output
        )

        ffn_output = self.dropout3(
            ffn_output
        )

        decoder_output = self.norm3(
            decoder_output + ffn_output
        )

        return decoder_output

In [16]:
import torch
import torch.nn as nn
import math

# TransformerEncoder (already in b8d8e772, moved here for order)
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, src, src_mask):
        x = self.embedding(src)
        x = self.positional_encoding(x)
        for layer in self.encoder_layers:
            x = layer(x, mask=src_mask)
        return x

# TransformerDecoder (from cell p55tJHues3fz)
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, tgt, enc_output, src_mask, tgt_mask):
        x = self.embedding(tgt)
        x = self.positional_encoding(x)
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x

# FullTransformer (already in b8d8e772)
class FullTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, num_decoder_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(
            src_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, max_seq_len, dropout_rate
        )
        self.decoder = TransformerDecoder(
            tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_decoder_layers, max_seq_len, dropout_rate
        )
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        output = self.output_linear(dec_output)
        return output

# Model Parameters (re-using existing values from the notebook where appropriate)
src_vocab_size_full_transformer = vocab_size
tgt_vocab_size_full_transformer = vocab_size
num_layers = 2
num_heads = 2
ffn_hidden_dim = 32
num_encoder_layers_full_transformer = num_layers
num_decoder_layers_full_transformer = num_layers
dropout_rate_full_transformer = 0.1

# Instantiate the FullTransformer model
full_transformer_model = FullTransformer(
    src_vocab_size=src_vocab_size_full_transformer,
    tgt_vocab_size=tgt_vocab_size_full_transformer,
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_encoder_layers=num_encoder_layers_full_transformer,
    num_decoder_layers=num_decoder_layers_full_transformer,
    max_seq_len=max_seq_len,
    dropout_rate=dropout_rate_full_transformer
)

print("FullTransformer instance created successfully!")
print(full_transformer_model)

FullTransformer instance created successfully!
FullTransformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(39, 8)
    (positional_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder_layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (Wq): Linear(in_features=8, out_features=8, bias=True)
        (Wk): Linear(in_features=8, out_features=8, bias=True)
        (Wv): Linear(in_features=8, out_features=8, bias=True)
        (Wo): Linear(in_features=8, out_features=8, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=8, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=8, bias=True)
        )
        (ffn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=T